# Demo: End-to-End Inference
GNN-BERT Music Context Understanding — CSE425

Loads the trained Task 3 fusion model and runs inference on one MusicCaps
test clip end-to-end: audio -> graph, caption -> tokens, both through the
fusion model -> predicted tags.

Requires: `results/task3_fusion_model.pt` (produced by `05_task3_fusion.ipynb`)
and the corresponding processed MusicCaps graphs from `02_preprocessing.ipynb`.

In [1]:
import os, sys, json, ast
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "src"))

RAW = os.path.join(PROJECT_ROOT, "data/raw")
PROC = os.path.join(PROJECT_ROOT, "data/processed")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MC_GRAPH_DIR = os.path.join(PROC, "musiccaps_graphs")

print("Project root:", PROJECT_ROOT)

Project root: f:\BRACU\CSE425\Project\gnn-bert-music-context


In [2]:
import torch
import pandas as pd
from transformers import BertTokenizer
from torch_geometric.data import Batch

from fusion_model import GNNBERTFusion

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

d:\Program Files\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## 1. Load the trained Task 3 fusion model

In [3]:
mc = pd.read_csv(os.path.join(RAW, "musiccaps/musiccaps-public.csv"))
mc_graph_manifest = pd.read_csv(os.path.join(PROC, "musiccaps_graph_manifest.csv"))
successful_ids = set(mc_graph_manifest[mc_graph_manifest["success"] == True]["ytid"])
mc_final = mc[mc["ytid"].isin(successful_ids)].reset_index(drop=True)
mc_final["aspect_parsed"] = mc_final["aspect_list"].apply(ast.literal_eval)

from collections import Counter
all_aspects = [a for aspects in mc_final["aspect_parsed"] for a in aspects]
top30_aspects = [phrase for phrase, count in Counter(all_aspects).most_common(30)]
print("Loaded tag vocabulary:", len(top30_aspects), "tags")

Loaded tag vocabulary: 30 tags


In [4]:
model = GNNBERTFusion(num_tags=30).to(device)
model.load_state_dict(torch.load(os.path.join(RESULTS_DIR, "task3_fusion_model.pt"),
                                  map_location=device))
model.eval()
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print("Model loaded and ready for inference.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4674.61it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\zawad\AppData\Local\Temp\ipykernel_29624\1298022874.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pick

Model loaded and ready for inference.


## 2. Pick one clip and run inference end-to-end

In [5]:
for tag in top30_aspects:
    mc_final[tag] = mc_final["aspect_parsed"].apply(lambda a: 1 if tag in a else 0)

sample = mc_final.iloc[0]
print("YouTube ID:", sample["ytid"])
print("Caption:", sample["caption"])
print("True tags:", [t for t in top30_aspects if sample[t] == 1])

YouTube ID: -0Gj8-vB1q4
Caption: The low quality recording features a ballad song that contains sustained strings, mellow piano melody and soft female vocal singing over it. It sounds sad and soulful, like something you would hear at Sunday services.
True tags: ['low quality']


In [ ]:
graph = torch.load(os.path.join(MC_GRAPH_DIR, f"{sample['ytid']}.pt"), weights_only=False)
graph_batch = Batch.from_data_list([graph]).to(device)
print("Graph:", graph)


tokens = tokenizer(sample["caption"], padding="max_length", truncation=True,
                    max_length=128, return_tensors="pt")
input_ids = tokens["input_ids"].to(device)
attention_mask = tokens["attention_mask"].to(device)

Graph: Data(x=[5, 12], edge_index=[2, 16])


In [7]:
with torch.no_grad():
    logits = model(graph_batch, input_ids, attention_mask)
    probs = torch.sigmoid(logits).cpu().numpy()[0]

predicted_tags = [top30_aspects[i] for i in range(30) if probs[i] > 0.5]
print("Predicted tags:", predicted_tags)
print()
print("Per-tag probabilities (top 10):")
top10_idx = probs.argsort()[-10:][::-1]
for i in top10_idx:
    print(f"  {top30_aspects[i]:25s} {probs[i]:.3f}")

Predicted tags: ['low quality']

Per-tag probabilities (top 10):
  low quality               0.894
  pop                       0.035
  emotional                 0.027
  noisy                     0.015
  groovy bass               0.010
  passionate                0.006
  shimmering hi hats        0.004
  slow tempo                0.003
  groovy                    0.002
  piano                     0.002


## 3. a different clip

Change the index below to run inference on any other test clip (valid range: 0 to len(mc_final)-1).

In [8]:
def predict_tags(clip_idx):
    row = mc_final.iloc[clip_idx]
    graph = torch.load(os.path.join(MC_GRAPH_DIR, f"{row['ytid']}.pt"), weights_only=False)
    graph_batch = Batch.from_data_list([graph]).to(device)
    tokens = tokenizer(row["caption"], padding="max_length", truncation=True,
                        max_length=128, return_tensors="pt")
    with torch.no_grad():
        logits = model(graph_batch, tokens["input_ids"].to(device), tokens["attention_mask"].to(device))
        probs = torch.sigmoid(logits).cpu().numpy()[0]
    predicted = [top30_aspects[i] for i in range(30) if probs[i] > 0.5]
    true = [t for t in top30_aspects if row[t] == 1]
    print("Caption:", row["caption"])
    print("True tags:", true)
    print("Predicted tags:", predicted)

predict_tags(5)  # try changing this index

Caption: This clip is three tracks playing consecutively. The first one is an electric guitar lead harmony with a groovy bass line, followed by white noise and then a female vocalisation to a vivacious melody with a keyboard harmony, slick drumming, funky bass lines and male backup. The three songs are unrelated and unsynced.
True tags: ['instrumental', 'bass guitar']
Predicted tags: ['instrumental', 'bass guitar']
